# Download and read a sample

This tutorial fetches one acquisition, **HD-002** (Visium HD, mouse ovary), and walks through
the three files you will use most: the sample record, the raw-count H5AD and the per-cell index.

**You will learn to**

- download a single sample package from Hugging Face
- read the sample record and its context fields
- open the expression matrix without loading it into memory
- link a cell in the index to its matrix row

> **Code-only page.** Outputs appear when you run the notebook on the downloaded sample.

> **Requirements.** `pip install huggingface_hub anndata pyarrow pandas` · disk space for one sample package (images included).

## 1. Download

The dataset ships a helper script. It writes the sample under `smcc_data/<platform>/samples/<id>/`.

In [ ]:
!pip install -q huggingface_hub anndata pyarrow pandas
!hf download zhang-lab/sMCC40M examples/download_sample.py --repo-type dataset --local-dir .
!python examples/download_sample.py --repo zhang-lab/sMCC40M --platform HD --sample HD-002

Remove the leading `#` to run those three lines in a notebook, or run them in a terminal without `!`.
Each sample package also contains a self-check you can run at any time:

```bash
python smcc_data/HD/samples/HD-002/read_example.py
```

In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import anndata as ad

SAMPLE_ROOT = Path("smcc_data/HD/samples/HD-002")
assert SAMPLE_ROOT.exists(), "Run the download cell first"

## 2. The sample record

In a **sample-only** package, asset paths in `metadata/samples.parquet` are relative to the
acquisition folder (in the full release they are relative to the release root).

In [ ]:
records = pq.read_table(SAMPLE_ROOT / "metadata/samples.parquet").to_pylist()
assert len(records) == 1
sample = records[0]

pd.Series({k: sample.get(k) for k in [
    "sample_id", "platform", "assay", "species", "tissue", "stain", "segmentation_kind",
    "expression_file", "genes_file", "image_file", "mask_file", "source_url",
]})

Context fields (diagnosis, condition, age, sex, study) come with a matching `*_status` column, so
"not reported" is never confused with "healthy". Show every column that exists in your copy:

In [ ]:
context = {k: v for k, v in sample.items()
           if any(s in k for s in ("diagnosis", "condition", "age", "sex", "study", "tumor"))}
pd.Series(context)

## 3. The expression matrix

The H5AD holds **raw non-negative integer counts** in CSR format on the original measured feature
axis. Open it *backed* to inspect it without reading the matrix.

In [ ]:
adata = ad.read_h5ad(SAMPLE_ROOT / sample["expression_file"], backed="r")
print(adata)
print("obsm:", list(adata.obsm.keys()))   # spatial (pixels) and spatial_um (microns)

The feature table is stored next to the matrix and is aligned row-for-row with `adata.var`.
Not every feature is a gene: keep **Gene Expression** separate from controls.

In [ ]:
genes = pq.read_table(SAMPLE_ROOT / sample["genes_file"]).to_pandas()
assert len(genes) == adata.n_vars
print(genes.columns.tolist())
genes["feature_type"].value_counts()

## 4. Link a cell to its matrix row

The per-cell index lists **every** segmentation candidate. `h5ad_row` is empty for candidates
without RNA: that means *no matrix row*, not an all-zero cell.

In [ ]:
cells = pq.read_table(SAMPLE_ROOT / "metadata/cells.parquet").to_pandas()
print(f"{len(cells):,} candidates, {cells['h5ad_row'].notna().sum():,} with an expression row")
cells.head()

In [ ]:
cell = cells.dropna(subset=["h5ad_row"]).iloc[0]
row = int(cell["h5ad_row"])
assert str(adata.obs_names[row]) == str(cell["cell_id"]), "cell_id / row mismatch"

x = adata.X[row:row + 1]                       # one sparse row
nz = x.indices[:10]
pd.DataFrame({"gene_id": genes["gene_id"].iloc[nz].values,
              "feature_type": genes["feature_type"].iloc[nz].values,
              "count": x.data[:10]})

In [ ]:
adata.file.close()

## Next

[Tutorial 2](02_explore_acquisition.ipynb) loads HD-002 into memory and runs a standard
single-cell workflow on it.